# LUT Dependence Visualization

Explore which FPGA-build configuration parameters drive `real_LUT` (real, post-synthesis LUT usage), as a first step toward picking a regression model for `hardware/mvau_lut_calibration_dataset.csv`.

For every plot below, **y = `real_LUT`**. The x-axis panel shows each configuration parameter **independently** (one miniature scatter plot per parameter), not yet combined into a joint model.

The loading/plotting code is written generically (`DATASET_PATH`, `plot_param_panel`) so the same notebook can later be pointed at other calibration CSVs with the same schema (e.g. an alpha025-specific dataset) just by changing `DATASET_PATH` and re-running.

In [1]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


def _find_repo_root(start: Path) -> Path:
    """Walk up from cwd looking for the repo root, so this notebook works the
    same whether Jupyter's cwd is the repo root or analysis/hardware_calibration/ itself."""
    for candidate in [start, *start.parents]:
        if (candidate / "hardware").exists() and (candidate / "analysis").exists():
            return candidate
    return start


REPO_ROOT = _find_repo_root(Path.cwd())

# Point this at a different calibration CSV (same schema) to reuse the whole notebook.
DATASET_PATH = REPO_ROOT / "hardware" / "mvau_lut_calibration_dataset.csv"

sns.set_theme(style="whitegrid")

## Load Dataset

In [2]:
df = pd.read_csv(DATASET_PATH)
print(f"loaded {DATASET_PATH.relative_to(REPO_ROOT)}: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

loaded hardware\mvau_lut_calibration_dataset.csv: 89 rows, 29 columns


,partition,node_name,op_type,MH,MW,PE,SIMD,PE_folding_json,SIMD_folding_json,pe_simd_landed_matches_json,...,PE_times_SIMD,log2_MW,real_LUT,real_LUTRAM,real_SRL,real_FF,real_BRAM36,real_BRAM18,real_URAM,real_DSP
0,0,MVAU_hls_0,MVAU_hls,3,9,3,9,3,9,True,...,27,3.17,6097,0,0,1336,0,0,0,27
1,1,MVAU_hls_0,MVAU_hls,4,16,4,4,4,4,True,...,16,4.00,7912,25,0,616,0,0,0,16
2,1,MVAU_hls_1,MVAU_hls,16,4,4,4,4,4,True,...,16,2.00,2041,0,0,500,2,0,0,16
3,1,MVAU_hls_2,MVAU_hls,4,36,1,18,1,18,True,...,18,5.17,3414,0,0,1342,2,0,0,22
4,1,MVAU_hls_3,MVAU_hls,16,4,16,1,16,1,True,...,16,2.00,30961,0,0,1203,1,0,0,16


## Inspect Dataset Structure

Column dtypes and a check for missing values, before deciding which columns are usable numeric features vs. categorical ones.

In [3]:
print(df.dtypes)

missing = df.isna().sum()
missing = missing[missing > 0]
print("\nmissing values per column:" if len(missing) else "\nno missing values")
missing

partition                        int64
node_name                       object
op_type                         object
MH                               int64
MW                               int64
PE                               int64
SIMD                             int64
PE_folding_json                  int64
SIMD_folding_json                int64
pe_simd_landed_matches_json       bool
weightDataType                  object
inputDataType                   object
outputDataType                  object
weight_bits                      int64
act_bits                         int64
resType                         object
force_dsp                         bool
ram_style                       object
mem_mode                        object
PE_times_SIMD                    int64
log2_MW                        float64
real_LUT                         int64
real_LUTRAM                      int64
real_SRL                         int64
real_FF                          int64
real_BRAM36              

Series([], dtype: int64)

## Identify Feature and Target Columns

Target is `real_LUT`. Features are split into the numeric build-config parameters and the categorical/boolean ones. Note the other `real_*` columns (`real_LUTRAM`, `real_FF`, `real_DSP`, ...) are excluded from the feature list -- they're synthesis *outputs*, like `real_LUT` itself, not configuration inputs.

In [4]:
Y_COL = "real_LUT"

NUMERIC_PARAMS = [
    "MH", "MW", "PE", "SIMD",
    "PE_times_SIMD", "weight_bits", "act_bits",
]
CATEGORICAL_PARAMS: list[str] = []
ALL_PARAMS = NUMERIC_PARAMS + CATEGORICAL_PARAMS

# Per-parameter trend-line shape for plot_param_panel/the correlation table --
# defaults to "linear" for anything not listed here. SIMD gets "exp_decay"
# (fit log2(real_LUT) ~ a*SIMD + b, i.e. real_LUT ~ 2**b * 2**(a*SIMD)) instead
# of a straight line -- a plain linear fit forces one global slope across the
# whole SIMD range, but the real relationship plausibly decays similarly to
# 2**(-SIMD) (each extra unit of SIMD folding a smaller MULTIPLICATIVE bite
# out of remaining real_LUT, not a fixed additive one) rather than a straight
# line down to (and past) zero.
TREND_TYPE: dict[str, str] = {
    "SIMD": "exp_decay",
}

print(f"target: {Y_COL}")
print(f"{len(NUMERIC_PARAMS)} numeric params, {len(CATEGORICAL_PARAMS)} categorical params")
print(f"non-linear trend fits: {TREND_TYPE}")

## Compute Summary Statistics

In [5]:
df[NUMERIC_PARAMS + [Y_COL]].describe().T

,count,mean,std,min,25%,50%,75%,max
partition,89.0,3.348315,1.919405,0.0,2.0,3.0,5.00,7.00
MH,89.0,12.696629,10.423318,1.0,4.0,8.0,16.00,32.00
MW,89.0,28.775281,24.254218,1.0,8.0,16.0,36.00,72.00
PE,89.0,6.797753,7.234984,1.0,2.0,4.0,8.00,32.00
SIMD,89.0,7.808989,7.889689,1.0,2.0,4.0,9.00,36.00
PE_times_SIMD,89.0,24.685393,13.938661,4.0,16.0,16.0,32.00,80.00
log2_MW,89.0,4.233258,1.448144,0.0,3.0,4.0,5.17,6.17
weight_bits,89.0,5.415730,1.232228,4.0,4.0,6.0,6.00,8.00
act_bits,89.0,5.865169,1.341565,4.0,4.0,6.0,6.00,8.00
real_LUT,89.0,7836.651685,11043.518326,282.0,1172.0,2325.0,8360.00,51099.00


## Create Scatter Plot Panel (Feature vs LUT)

`plot_param_panel` builds one miniature scatter plot per parameter, always with `real_LUT` on the y-axis (log-scaled by default, since it spans multiple orders of magnitude):
- **Numeric** params (`MH`, `PE`, `weight_bits`, ...) get a plain scatter plus a linear trend line and a Pearson `r` annotation (see next two sections).
- **Categorical/boolean** params (`ram_style`, `op_type`, ...) get their category codes jittered slightly on the x-axis, with the real category names as tick labels.

Every point is colored by `op_type` (e.g. `MVAU_hls` vs `VVAU_hls`), with a single shared legend for the whole figure.

In [6]:
def plot_param_panel(
    df: pd.DataFrame,
    params: list[str],
    y_col: str = Y_COL,
    y_log: bool = False,
    ncols: int = 4,
    add_trend: bool = True,
    title: str | None = None,
) -> plt.Figure:
    """Grid of miniature scatter plots, one per parameter in `params`, always
    plotting df[y_col] on the y-axis. Numeric params (per NUMERIC_PARAMS) get
    a trend line + correlation `r` in the subplot title -- straight-line
    linear by default, or an exp_decay curve (log2(y) ~ a*x+b) for any
    parameter listed in TREND_TYPE, e.g. SIMD. Everything else is treated as
    categorical: jittered integer category codes on the x-axis, with the real
    category names as tick labels. Every point is colored by `op_type` (kept
    purely as a visual aid even though op_type itself was dropped from
    ALL_PARAMS -- distinguishes MVAU_hls/VVAU_hls/MVAU_rtl rows at a glance
    without treating op_type as a feature being analyzed), with the legend
    placed OUTSIDE the axes (bbox_to_anchor) so it never overlaps a subplot
    title regardless of how long that title is. y_log=False (linear y-axis)
    for now."""
    op_types = sorted(df["op_type"].astype(str).unique())
    cmap = plt.get_cmap("tab10")
    color_map = {op: cmap(i) for i, op in enumerate(op_types)}
    colors = df["op_type"].astype(str).map(color_map)

    nrows = int(np.ceil(len(params) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 3.0 * nrows), squeeze=False)
    axes = axes.ravel()
    rng = np.random.default_rng(0)
    y = df[y_col].astype(float)

    for ax, param in zip(axes, params):
        if param in NUMERIC_PARAMS:
            x = df[param].astype(float)
            ax.scatter(x, y, c=colors, s=14, alpha=0.75, edgecolors="none")
            trend_type = TREND_TYPE.get(param, "linear")

            if trend_type == "exp_decay":
                # log2(y) ~ a*x + b  <=>  y ~ 2**b * 2**(a*x) -- linearize in
                # log-space so a plain polyfit(deg=1) still works, then map
                # the fitted line back through 2**(...) for plotting. This is
                # just the fitting technique for the exponential-decay SHAPE
                # requested for SIMD -- unrelated to the (now removed) y-axis
                # log-scale/log2_MW feature.
                log2_y = np.log2(y)
                if add_trend and x.nunique() > 1:
                    coeffs = np.polyfit(x, log2_y, deg=1)
                    x_line = np.linspace(x.min(), x.max(), 50)
                    ax.plot(x_line, 2 ** np.polyval(coeffs, x_line), color="black", linewidth=1, linestyle="--")
                    rate = coeffs[0]
                else:
                    rate = float("nan")
                r = np.corrcoef(x, log2_y)[0, 1] if x.nunique() > 1 else float("nan")
                ax.set_title(f"{param}  (exp: rate={rate:.2f}, r={r:.2f})", fontsize=9)
            else:
                if add_trend and x.nunique() > 1:
                    coeffs = np.polyfit(x, y, deg=1)
                    x_line = np.linspace(x.min(), x.max(), 50)
                    ax.plot(x_line, np.polyval(coeffs, x_line), color="black", linewidth=1, linestyle="--")
                r = np.corrcoef(x, y)[0, 1] if x.nunique() > 1 else float("nan")
                ax.set_title(f"{param}  (r={r:.2f})", fontsize=9)
        else:
            cats = sorted(df[param].astype(str).unique())
            code_of = {c: i for i, c in enumerate(cats)}
            codes = df[param].astype(str).map(code_of).astype(float)
            jitter = rng.uniform(-0.15, 0.15, size=len(codes))
            ax.scatter(codes + jitter, y, c=colors, s=14, alpha=0.75, edgecolors="none")
            ax.set_xticks(range(len(cats)))
            ax.set_xticklabels(cats, rotation=45, ha="right", fontsize=7)
            ax.set_title(param, fontsize=9)

        ax.set_xlabel(param, fontsize=8)
        ax.tick_params(labelsize=7)
        if y_log:
            ax.set_yscale("log")

    for ax in axes[len(params):]:
        ax.axis("off")
    for row in range(nrows):
        axes[row * ncols].set_ylabel(y_col, fontsize=8)

    handles = [
        plt.Line2D([0], [0], marker="o", linestyle="", color=color_map[op], label=op)
        for op in op_types
    ]
    fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=9, title="op_type")

    if title:
        fig.suptitle(title, fontsize=13, y=1.02)
    fig.tight_layout()
    return fig

In [7]:
FEATURES_PANEL_PARAMS = [p for p in ALL_PARAMS if p != "PE_times_SIMD"]  # dropped for now -- redundant with PE/SIMD shown separately

fig = plot_param_panel(
    df, FEATURES_PANEL_PARAMS,
    title=f"{DATASET_PATH.name}: real_LUT vs each parameter",
)
plt.show()

## Highlight Correlation Strength per Feature

The trend lines/`r` values above are read off each subplot title; here they're collected into one sorted table (by `|r|`, strongest first) for easier comparison across all numeric parameters at once.

In [8]:
y = df[Y_COL].astype(float)
corr_rows = []
for param in NUMERIC_PARAMS:
    x = df[param].astype(float)
    trend_type = TREND_TYPE.get(param, "linear")
    if trend_type == "exp_decay":
        # Same log2(y)-vs-x correlation the panel plot's exp_decay trend line
        # is fit against -- comparing this r side-by-side with the other
        # (plain linear) params' r is apples-to-oranges in absolute terms,
        # but tells you how well EACH param's own best-suited trend shape
        # explains it, which is the more useful question here.
        r = np.corrcoef(x, np.log2(y))[0, 1] if x.nunique() > 1 else float("nan")
    else:
        r = np.corrcoef(x, y)[0, 1] if x.nunique() > 1 else float("nan")
    corr_rows.append({"parameter": param, "trend_type": trend_type, "pearson_r": r, "abs_r": abs(r)})

corr_df = (
    pd.DataFrame(corr_rows)
    .sort_values("abs_r", ascending=False)
    .drop(columns="abs_r")
    .reset_index(drop=True)
)
corr_df

## Save Visualizations

In [9]:
RESULTS_DIR = REPO_ROOT / "analysis" / "hardware_calibration" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

out_path = RESULTS_DIR / f"{DATASET_PATH.stem}_lut_panel.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
print(f"saved panel to {out_path.relative_to(REPO_ROOT)}")

saved panel to analysis\hardware_calibration\results\mvau_lut_calibration_dataset_lut_panel.png


## Transforms

A different hypothesis than the "features" panel's own `exp_decay` option (which keeps x linear and log-transforms *y*): here the **x-value itself** is transformed before fitting a plain linear trend against untransformed `real_LUT`. `SIMD` and `MW` get `2^-x` (does `real_LUT` decay linearly with a multiplicative shrink in x?); `PE` and `MH` each get **two** transforms side by side, `log2(x)` (does `real_LUT` grow linearly with the *number of doublings* in x?) and `2^x` (the opposite hypothesis -- does `real_LUT` itself grow like a power of 2 in x, i.e. accelerate rather than compress?). Everything else (`weight_bits`, `act_bits`) stays plain linear, i.e. untransformed.

In [ ]:
TRANSFORM_SPECS: list[tuple[str, str, "callable"]] = [
    # (subplot label, source column, transform function)
    ("2^-SIMD", "SIMD", lambda x: 2.0 ** (-x)),
    ("2^-MW", "MW", lambda x: 2.0 ** (-x)),
    ("log2(PE)", "PE", np.log2),
    ("2^PE", "PE", lambda x: 2.0 ** x),
    ("log2(MH)", "MH", np.log2),
    ("2^MH", "MH", lambda x: 2.0 ** x),
    ("weight_bits", "weight_bits", lambda x: x),
    ("act_bits", "act_bits", lambda x: x),
]


def plot_transform_panel(
    df: pd.DataFrame,
    specs: list[tuple[str, str, "callable"]],
    y_col: str = Y_COL,
    y_log: bool = False,
    ncols: int = 4,
    title: str | None = None,
) -> plt.Figure:
    """Like plot_param_panel, but each entry in `specs` is (label, source
    column, transform function) rather than a bare parameter name -- lets a
    single source column (e.g. PE) appear more than once under different
    transforms (log2(PE) AND 2^PE side by side), which a one-transform-per-
    parameter dict can't express. Transforms the x-VALUE itself before
    scatter-plotting + fitting a plain linear trend against untransformed
    real_LUT -- a genuinely different hypothesis from plot_param_panel's own
    exp_decay option (which keeps x linear and log-transforms y instead)."""
    op_types = sorted(df["op_type"].astype(str).unique())
    cmap = plt.get_cmap("tab10")
    color_map = {op: cmap(i) for i, op in enumerate(op_types)}
    colors = df["op_type"].astype(str).map(color_map)

    nrows = int(np.ceil(len(specs) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 3.0 * nrows), squeeze=False)
    axes = axes.ravel()
    y = df[y_col].astype(float)

    for ax, (label, source_col, transform) in zip(axes, specs):
        x = transform(df[source_col].astype(float))
        ax.scatter(x, y, c=colors, s=14, alpha=0.75, edgecolors="none")

        if x.nunique() > 1:
            coeffs = np.polyfit(x, y, deg=1)
            x_line = np.linspace(x.min(), x.max(), 50)
            ax.plot(x_line, np.polyval(coeffs, x_line), color="black", linewidth=1, linestyle="--")
            r = np.corrcoef(x, y)[0, 1]
        else:
            r = float("nan")
        ax.set_title(f"{label}  (r={r:.2f})", fontsize=9)
        ax.set_xlabel(label, fontsize=8)
        ax.tick_params(labelsize=7)
        if y_log:
            ax.set_yscale("log")

    for ax in axes[len(specs):]:
        ax.axis("off")
    for row in range(nrows):
        axes[row * ncols].set_ylabel(y_col, fontsize=8)

    handles = [
        plt.Line2D([0], [0], marker="o", linestyle="", color=color_map[op], label=op)
        for op in op_types
    ]
    fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=9, title="op_type")

    if title:
        fig.suptitle(title, fontsize=13, y=1.02)
    fig.tight_layout()
    return fig

In [ ]:
fig_transform = plot_transform_panel(
    df, TRANSFORM_SPECS,
    title=f"{DATASET_PATH.name}: real_LUT vs transformed parameters",
)
plt.show()

In [ ]:
out_path_transform = RESULTS_DIR / f"{DATASET_PATH.stem}_lut_transform_panel.png"
fig_transform.savefig(out_path_transform, dpi=150, bbox_inches="tight")
print(f"saved transform panel to {out_path_transform.relative_to(REPO_ROOT)}")

## Interactions

Engineered multi-column features (not single-parameter transforms) plotted the same way as the panels above -- one point per node, real_LUT on y, plain linear trend + Pearson `r`. `W`/`A` below are `weight_bits`/`act_bits`.

In [ ]:
INTERACTION_SPECS: list[tuple[str, "callable"]] = [
    ("PE*SIMD", lambda d: d["PE"] * d["SIMD"]),
    ("PE/SIMD", lambda d: d["PE"] / d["SIMD"]),
    ("PE*(A+W)", lambda d: d["PE"] * (d["act_bits"] + d["weight_bits"])),
    ("PE*MH", lambda d: d["PE"] * d["MH"]),
    ("PE/MW", lambda d: d["PE"] / d["MW"]),
    ("PE-SIMD", lambda d: d["PE"] - d["SIMD"]),
    ("PE*MH - SIMD*MW", lambda d: d["PE"] * d["MH"] - d["SIMD"] * d["MW"]),
    ("(PE*MH - SIMD*MW)*(W+A)", lambda d: (d["PE"] * d["MH"] - d["SIMD"] * d["MW"]) * (d["weight_bits"] + d["act_bits"])),
    ("(log2(PE)/2^-SIMD)*(A+W)", lambda d: (np.log2(d["PE"]) / (2.0 ** (-d["SIMD"]))) * (d["act_bits"] + d["weight_bits"])),
]


def plot_interaction_panel(
    df: pd.DataFrame,
    specs: list[tuple[str, "callable"]],
    y_col: str = Y_COL,
    y_log: bool = False,
    ncols: int = 4,
    title: str | None = None,
) -> plt.Figure:
    """Like plot_transform_panel, but each spec is (label, function-of-the-
    WHOLE-dataframe) rather than (label, source column, function-of-one-
    column) -- lets a spec combine multiple columns (PE*SIMD, PE*MH -
    SIMD*MW, ...) instead of transforming a single one. Plain linear trend +
    Pearson r against untransformed real_LUT, same styling as the other
    panels."""
    op_types = sorted(df["op_type"].astype(str).unique())
    cmap = plt.get_cmap("tab10")
    color_map = {op: cmap(i) for i, op in enumerate(op_types)}
    colors = df["op_type"].astype(str).map(color_map)

    nrows = int(np.ceil(len(specs) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 3.0 * nrows), squeeze=False)
    axes = axes.ravel()
    y = df[y_col].astype(float)

    for ax, (label, compute) in zip(axes, specs):
        x = compute(df).astype(float)
        ax.scatter(x, y, c=colors, s=14, alpha=0.75, edgecolors="none")

        if x.nunique() > 1:
            coeffs = np.polyfit(x, y, deg=1)
            x_line = np.linspace(x.min(), x.max(), 50)
            ax.plot(x_line, np.polyval(coeffs, x_line), color="black", linewidth=1, linestyle="--")
            r = np.corrcoef(x, y)[0, 1]
        else:
            r = float("nan")
        ax.set_title(f"{label}  (r={r:.2f})", fontsize=9)
        ax.set_xlabel(label, fontsize=8)
        ax.tick_params(labelsize=7)
        if y_log:
            ax.set_yscale("log")

    for ax in axes[len(specs):]:
        ax.axis("off")
    for row in range(nrows):
        axes[row * ncols].set_ylabel(y_col, fontsize=8)

    handles = [
        plt.Line2D([0], [0], marker="o", linestyle="", color=color_map[op], label=op)
        for op in op_types
    ]
    fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(1.0, 1.0), fontsize=9, title="op_type")

    if title:
        fig.suptitle(title, fontsize=13, y=1.02)
    fig.tight_layout()
    return fig

In [ ]:
fig_interaction = plot_interaction_panel(
    df, INTERACTION_SPECS,
    title=f"{DATASET_PATH.name}: real_LUT vs interaction features",
)
plt.show()

In [ ]:
out_path_interaction = RESULTS_DIR / f"{DATASET_PATH.stem}_lut_interaction_panel.png"
fig_interaction.savefig(out_path_interaction, dpi=150, bbox_inches="tight")
print(f"saved interaction panel to {out_path_interaction.relative_to(REPO_ROOT)}")

## Quick Test: Any X-Axis Function

A single-plot utility for trying a NEW x-axis expression on the fly, without editing `INTERACTION_SPECS`/`TRANSFORM_SPECS` or re-running a whole multi-panel figure. `quick_test` takes any function of the dataframe (or of a single column, via the `col=` shortcut) and immediately plots + fits + reports `r` -- the fast iteration loop for "what if I try X" questions like the last few cells above.

In [ ]:
def quick_test(fn: "callable", label: str | None = None, col: str | None = None,
               df: pd.DataFrame = df, y_col: str = Y_COL) -> float:
    """Single-plot ad hoc test for a NEW x-axis expression -- pass any
    function of the WHOLE dataframe (fn(df) -> array-like, for multi-column
    formulas like interactions) or use col= as a shortcut to apply fn to
    just one column (fn(df[col]) -> array-like, for single-parameter
    transforms). Immediately plots a scatter + linear trend + Pearson r and
    returns r, no need to edit INTERACTION_SPECS/TRANSFORM_SPECS or re-run a
    whole multi-panel figure for a one-off idea."""
    x = fn(df[col].astype(float)) if col is not None else fn(df)
    x = pd.Series(x, index=df.index).astype(float)
    y = df[y_col].astype(float)

    op_types = sorted(df["op_type"].astype(str).unique())
    cmap = plt.get_cmap("tab10")
    color_map = {op: cmap(i) for i, op in enumerate(op_types)}
    colors = df["op_type"].astype(str).map(color_map)

    fig, ax = plt.subplots(figsize=(5.5, 4.5))
    ax.scatter(x, y, c=colors, s=20, alpha=0.75, edgecolors="none")

    r = float("nan")
    if x.nunique() > 1:
        coeffs = np.polyfit(x, y, deg=1)
        x_line = np.linspace(x.min(), x.max(), 50)
        ax.plot(x_line, np.polyval(coeffs, x_line), color="black", linewidth=1, linestyle="--")
        r = np.corrcoef(x, y)[0, 1]

    label = label or "x"
    ax.set_title(f"{label}  (r={r:.3f})", fontsize=11)
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel(y_col, fontsize=10)
    handles = [plt.Line2D([0], [0], marker="o", linestyle="", color=color_map[op], label=op) for op in op_types]
    ax.legend(handles=handles, fontsize=8, loc="best")
    fig.tight_layout()
    plt.show()
    return r

In [ ]:
quick_test(lambda d: (d["PE"] / d["SIMD"]) * (d["act_bits"] + d["weight_bits"]), label="PE/SIMD*(A+W)")

In [ ]:
quick_test(lambda d: (np.log2(d["PE"]) + 2.0 ** (-d["SIMD"])) * (d["act_bits"] + d["weight_bits"]), label="(log2(PE)+2^-SIMD)*(A+W)")

In [ ]:
quick_test(
    lambda d: np.minimum(0, d["PE"] - d["SIMD"]) + np.minimum(0, d["MH"] - d["MW"]),
    label="min(0,PE-SIMD)+min(0,MH-MW)",
)

In [ ]:
# User's observation on outliers.csv, confirmed directly: the top-10 worst
# under-predictions ALL have PE/SIMD>=8 AND MH/MW>=2 (mostly exactly 4)
# simultaneously; the worst over-predictions are the mirror image (both
# ratios small). Both ratios respond in the SAME direction together, not
# independently -- sum captures that better than product or max here.
quick_test(
    lambda d: (d["PE"] / d["SIMD"] + d["MH"] / d["MW"]) * (d["act_bits"] + d["weight_bits"]),
    label="(PE/SIMD+MH/MW)*(A+W)",
)

## Predicted + Imbalance vs. Real LUT (Updated Parity Plot)

Same style as the earlier "Cost-Model Prediction vs. Real LUT" parity plot, but now WITH the refit `imbalance_luts` term added in -- this is `compression/hawq/finn_cost_model.py`'s actual live formula now, not the physically-derived baseline alone.

`c0`/`c1` stay at their real-FINN-source-derived defaults (`300`, `1.1`) -- a joint refit alongside the imbalance term was tried and only moved R^2 from 0.636 to 0.638, not worth losing their source provenance for. The imbalance term itself uses **separate** coefficients for its two ratios rather than one shared `k`: `imbalance_luts = (k_pe*(PE/SIMD) + k_mh*(MH/MW)) * (A+W)`, with `k_pe=92.54`, `k_mh=155.84` (R^2=0.644, vs 0.636 for a single shared `k=104.56`) -- `k_mh > k_pe` says an `MH/MW` imbalance unit costs more real LUT than a `PE/SIMD` one. A further 4-way split (separating the `(A+W)` scaling into per-ratio `A` and `W` terms) reached R^2=0.675 but produced a physically-nonsensical NEGATIVE coefficient (`PE/SIMD * weight_bits` = -102.31) -- rejected as overfitting 4 free parameters on 89 noisy rows.

In [ ]:
def cost_model_lut_with_imbalance(pe, simd, w, a, mw, mh):
    """finn_cost_model.py's current LIVE conv_cost_pe_simd formula, in full --
    baseline (swu_lut + c0 + c1*P*(addertree_luts+acc_luts)) PLUS the refit
    imbalance_luts term. c0/c1 stay at their FINN-source-derived defaults
    (300, 1.1); the imbalance term uses separate coefficients for its two
    ratios (k_pe for PE/SIMD, k_mh for MH/MW), matching finn_cost_model.py."""
    addertree_luts = (w + a) * (2 * simd - 1)
    alpha = np.log2(mw) + w + a - 2
    acc_luts = np.minimum(32, alpha + np.log2(1 + 2.0 ** -alpha) + 1)
    c0, c1 = 300, 1.1
    k_pe, k_mh = 92.5386, 155.8369
    imbalance_luts = (k_pe * (pe / simd) + k_mh * (mh / mw)) * (w + a)
    mvu_lut = c0 + c1 * pe * (addertree_luts + acc_luts) + imbalance_luts
    return 426 + mvu_lut


df["predicted_lut"] = cost_model_lut_with_imbalance(
    df["PE"], df["SIMD"], df["weight_bits"], df["act_bits"], df["MW"], df["MH"],
)

op_types = sorted(df["op_type"].astype(str).unique())
cmap = plt.get_cmap("tab10")
color_map = {op: cmap(i) for i, op in enumerate(op_types)}
colors = df["op_type"].astype(str).map(color_map)

fig_parity2, ax = plt.subplots(figsize=(6.5, 6.5))
ax.scatter(df["predicted_lut"], df["real_LUT"], c=colors, s=28, alpha=0.8, edgecolors="none")

lims = [0, max(df["predicted_lut"].max(), df["real_LUT"].max()) * 1.05]
ax.plot(lims, lims, color="black", linewidth=1, linestyle="--", label="y = x (perfect prediction)")
ax.set_xlim(lims)
ax.set_ylim(lims)

r = np.corrcoef(df["predicted_lut"], df["real_LUT"])[0, 1]
ss_res = np.sum((df["real_LUT"] - df["predicted_lut"]) ** 2)
ss_tot = np.sum((df["real_LUT"] - df["real_LUT"].mean()) ** 2)
r2 = 1 - ss_res / ss_tot
ax.set_xlabel("predicted_lut  (finn_cost_model.py, WITH imbalance_luts)", fontsize=10)
ax.set_ylabel("real_LUT  (real post-synthesis)", fontsize=10)
ax.set_title(f"Predicted+imbalance vs. real LUT, per MVAU/VVAU node  (r={r:.2f}, R^2={r2:.2f})", fontsize=11)

handles = [plt.Line2D([0], [0], marker="o", linestyle="", color=color_map[op], label=op) for op in op_types]
handles.append(plt.Line2D([0], [0], color="black", linewidth=1, linestyle="--", label="y = x"))
ax.legend(handles=handles, fontsize=9, loc="upper left")
fig_parity2.tight_layout()
plt.show()

In [ ]:
out_path_parity2 = RESULTS_DIR / f"{DATASET_PATH.stem}_lut_parity_with_imbalance.png"
fig_parity2.savefig(out_path_parity2, dpi=150, bbox_inches="tight")
print(f"saved updated parity plot to {out_path_parity2.relative_to(REPO_ROOT)}")

## Mean LUT per Discrete Parameter Value

A separate panel from the raw-row scatter above: for each parameter, group rows by that parameter's own DISCRETE value (e.g. `act_bits` only ever takes on a handful of values like 4/6/8), and plot **mean `real_LUT`** per value instead of every individual row. Error bars show +/-1 std within the group, and each point is annotated with its group's row count (`n=`) so a mean resting on 1-2 rows isn't mistaken for a well-supported trend.

In [ ]:
def plot_param_mean_panel(
    df: pd.DataFrame,
    params: list[str],
    y_col: str = Y_COL,
    y_log: bool = False,
    ncols: int = 4,
    title: str | None = None,
) -> plt.Figure:
    """Grid of miniature plots, one per parameter in `params`, showing
    mean(df[y_col]) per DISCRETE value of that parameter (e.g. mean real_LUT
    at act_bits=4 vs 6 vs 8) instead of every raw row -- a cleaner look at
    each parameter's trend than plot_param_panel's per-row scatter above,
    especially for params with only a handful of distinct real values (bits,
    PE, SIMD, MH, ...). Error bars are +/-1 std within the group; each point
    is annotated with its group's row count so a mean resting on 1-2 rows
    isn't mistaken for a well-supported trend."""
    nrows = int(np.ceil(len(params) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.6 * ncols, 3.0 * nrows), squeeze=False)
    axes = axes.ravel()

    for ax, param in zip(axes, params):
        grouped = df.groupby(param)[y_col].agg(["mean", "std", "count"]).reset_index()
        grouped = grouped.sort_values(param).reset_index(drop=True)
        x_labels = grouped[param].astype(str)
        x_pos = np.arange(len(grouped))

        ax.errorbar(
            x_pos, grouped["mean"], yerr=grouped["std"].fillna(0),
            fmt="o-", color="tab:blue", ecolor="tab:blue", elinewidth=1, capsize=3, markersize=5,
        )
        for xi, (m, n) in enumerate(zip(grouped["mean"], grouped["count"])):
            ax.annotate(f"n={n}", (xi, m), textcoords="offset points", xytext=(0, 6),
                        fontsize=6, ha="center", color="gray")

        ax.set_xticks(x_pos)
        ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=7)
        ax.set_title(f"{param}", fontsize=9)
        ax.set_xlabel(param, fontsize=8)
        ax.tick_params(labelsize=7)
        if y_log:
            ax.set_yscale("log")

    for ax in axes[len(params):]:
        ax.axis("off")
    for row in range(nrows):
        axes[row * ncols].set_ylabel(f"mean {y_col}", fontsize=8)

    if title:
        fig.suptitle(title, fontsize=13, y=1.02)
    fig.tight_layout()
    return fig

In [ ]:
fig_mean = plot_param_mean_panel(
    df, ALL_PARAMS,
    title=f"{DATASET_PATH.name}: mean real_LUT per discrete parameter value",
)
plt.show()

In [ ]:
out_path_mean = RESULTS_DIR / f"{DATASET_PATH.stem}_lut_mean_panel.png"
fig_mean.savefig(out_path_mean, dpi=150, bbox_inches="tight")
print(f"saved mean panel to {out_path_mean.relative_to(REPO_ROOT)}")

## Cost-Model Prediction vs. Real LUT (Parity Plot)

A different kind of plot from everything above: instead of one parameter at a time, this puts `compression/hawq/finn_cost_model.py`'s own analytical LUT formula (the physically-derived part only -- `swu_lut + mvu_lut`, `force_dsp=True` so `mult_luts=0`, **no** empirical `imbalance_luts` correction added) on the x-axis and real post-synthesis `real_LUT` on the y-axis, one point per MVAU/VVAU node. The dashed line is `y=x` (perfect prediction) -- points above it are under-predicted by the model, points below are over-predicted. This is the actual discrepancy the whole `imbalance_luts` investigation exists to close, made visible directly rather than inferred from per-parameter panels.

In [ ]:
def cost_model_lut(pe: pd.Series, simd: pd.Series, w: pd.Series, a: pd.Series, mw: pd.Series) -> pd.Series:
    """finn_cost_model.py's conv_cost_pe_simd mvu_lut formula, physically-
    derived part only (swu_lut=426 flat + c0=300 + addertree_luts + acc_luts),
    force_dsp=True so mult_luts=0 -- matches every row in this dataset
    (force_dsp column is True throughout). Deliberately EXCLUDES the
    empirical imbalance_luts term -- this plot exists to show the gap that
    term is trying to close, not to already include it."""
    addertree_luts = (w + a) * (2 * simd - 1)
    alpha = np.log2(mw) + w + a - 2
    acc_luts = np.minimum(32, alpha + np.log2(1 + 2.0 ** -alpha) + 1)
    mvu_lut = 300 + 1.1 * pe * (addertree_luts + acc_luts)
    return 426 + mvu_lut


df["cost_model_lut"] = cost_model_lut(df["PE"], df["SIMD"], df["weight_bits"], df["act_bits"], df["MW"])

op_types = sorted(df["op_type"].astype(str).unique())
cmap = plt.get_cmap("tab10")
color_map = {op: cmap(i) for i, op in enumerate(op_types)}
colors = df["op_type"].astype(str).map(color_map)

fig_parity, ax = plt.subplots(figsize=(6.5, 6.5))
ax.scatter(df["cost_model_lut"], df["real_LUT"], c=colors, s=28, alpha=0.8, edgecolors="none")

# x-axis limited to the cost model's OWN largest prediction (not the larger
# of the two axes' maxima) -- the whole point of this plot is to show how
# compressed/narrow the model's predicted range is relative to real_LUT's
# actual spread, which a shared square x=y range would hide by leaving most
# of the plot's x-extent empty. The y=x line still gets drawn all the way up
# to real_LUT's own max and simply runs off the top of the visible x-range --
# that clipping IS the point, not a bug to hide.
x_max = df["cost_model_lut"].max() * 1.05
y_max = df["real_LUT"].max() * 1.05
ax.plot([0, y_max], [0, y_max], color="black", linewidth=1, linestyle="--", label="y = x (perfect prediction)")
ax.set_xlim(0, x_max)
ax.set_ylim(0, y_max)

r = np.corrcoef(df["cost_model_lut"], df["real_LUT"])[0, 1]
ax.set_xlabel("cost_model_lut  (finn_cost_model.py, no imbalance_luts)", fontsize=10)
ax.set_ylabel("real_LUT  (real post-synthesis)", fontsize=10)
ax.set_title(f"Cost-model prediction vs. real LUT, per MVAU/VVAU node  (r={r:.2f})", fontsize=11)

handles = [plt.Line2D([0], [0], marker="o", linestyle="", color=color_map[op], label=op) for op in op_types]
handles.append(plt.Line2D([0], [0], color="black", linewidth=1, linestyle="--", label="y = x"))
ax.legend(handles=handles, fontsize=9, loc="upper left")
fig_parity.tight_layout()
plt.show()

In [ ]:
out_path_parity = RESULTS_DIR / f"{DATASET_PATH.stem}_lut_parity.png"
fig_parity.savefig(out_path_parity, dpi=150, bbox_inches="tight")
print(f"saved parity plot to {out_path_parity.relative_to(REPO_ROOT)}")

## Outliers / Compliers: split by +/-20% band around `cost_model_lut`

Splits every MVAU/VVAU node into two groups relative to the parity plot's `y=x` line: **compliers** (`real_LUT` within a symmetric +/-20% band of `cost_model_lut`) and **outliers** (everything outside that band, in either direction -- under-predicted *or* over-predicted by more than 20%). Columns are the current feature panel's own parameters (`NUMERIC_PARAMS`) plus `real_LUT` and `cost_model_lut` themselves, so each row's actual vs. predicted values are visible alongside its features. Saved as `outliers.csv` and `compliers.csv`.

In [ ]:
BAND = 0.2  # symmetric +/-20% band around cost_model_lut
compliant_mask = df["real_LUT"].between((1 - BAND) * df["cost_model_lut"], (1 + BAND) * df["cost_model_lut"])

df["delta"] = df["real_LUT"] - df["cost_model_lut"]
SNIPPET_COLS = NUMERIC_PARAMS + ["real_LUT", "cost_model_lut", "delta"]
outliers = df.loc[~compliant_mask, SNIPPET_COLS]
compliers = df.loc[compliant_mask, SNIPPET_COLS]

print(f"outliers  (outside +/-{BAND:.0%} of cost_model_lut, either direction): {len(outliers)}/{len(df)} rows")
print(f"compliers (within +/-{BAND:.0%} of cost_model_lut): {len(compliers)}/{len(df)} rows")

outliers_path = RESULTS_DIR / "outliers.csv"
compliers_path = RESULTS_DIR / "compliers.csv"
outliers.to_csv(outliers_path)
compliers.to_csv(compliers_path)
print(f"saved {outliers_path.relative_to(REPO_ROOT)}")
print(f"saved {compliers_path.relative_to(REPO_ROOT)}")

outliers

## Reusing this notebook for other calibration CSVs

`plot_param_panel`, `NUMERIC_PARAMS`/`CATEGORICAL_PARAMS`, and the correlation-table cell all work off `df` alone -- to analyze another calibration dataset with the same schema (e.g. `hardware/mvau_lut_calibration_dataset_12_separable_dense_relu_alpha025.csv`), just change `DATASET_PATH` in the Load Dataset cell above, re-run from there, and everything else (structure inspection, panel plot, correlation table, saved PNG) regenerates for the new dataset.